# NB01: OPAM2 vs Marvin pKa Comparison

Compare OPAM2 (ModelSEED-finetuned GCN) pKa predictions against Marvin 23.4 (ChemAxon) values
stored in the ModelSEEDDatabase. This notebook loads existing benchmark results from OPAM2
and performs thermodynamics-focused analysis: which compounds have the largest pKa shifts,
how do shifts distribute by functional group, and which magnitude categories dominate.

**Key convention**: Marvin's `pkb` column contains pKa values for protonated basic sites,
NOT pKb values. No `14 - x` conversion should be applied.

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OPAM2_ROOT = Path('/tmp/OPAM2')
MSDB_ROOT = Path('/tmp/ModelSEEDDatabase')
PROJECT_ROOT = Path('..').resolve()
FIGURES_DIR = PROJECT_ROOT / 'figures'
DATA_DIR = PROJECT_ROOT / 'data'
FIGURES_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(OPAM2_ROOT / 'src'))

## 1. Load existing benchmark results

The OPAM2 benchmark (`benchmarks/run_modelseed_comparison.py`) has already been run.
It compared 23,854 compounds with .mol files, producing 82,171 matched atom-level
pKa comparisons.

In [ ]:
results_dir = OPAM2_ROOT / 'benchmarks' / 'results'

df_match = pd.read_csv(results_dir / 'modelseed_comparison.csv')
df_cpd = pd.read_csv(results_dir / 'modelseed_per_compound.csv')

with open(results_dir / 'modelseed_summary.json') as f:
    summary = json.load(f)

print(f"Compounds with .mol files: {summary['n_compounds_processed']:,}")
print(f"Compounds skipped (no .mol): {summary['n_compounds_skipped_no_mol']:,}")
print(f"Matched atoms: {summary['n_matched_atoms']:,}")
print(f"Overall R²: {summary['overall']['r2']:.3f}")
print(f"Overall RMSE: {summary['overall']['rmse']:.2f}")
print(f"Overall MAE: {summary['overall']['mae']:.2f}")
print(f"Median |diff|: {summary['overall']['median_abs_diff']:.2f}")

## 2. pKa difference distribution

In [ ]:
df_match['diff'] = df_match['molgpka_pka'] - df_match['marvin_pka']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of signed differences
axes[0].hist(df_match['diff'], bins=100, edgecolor='none', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)
axes[0].set_xlabel('OPAM2 pKa - Marvin pKa')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Signed pKa Difference (n={len(df_match):,})')

# CDF of absolute differences
sorted_abs = np.sort(df_match['abs_diff'].values)
cdf = np.arange(1, len(sorted_abs) + 1) / len(sorted_abs)
axes[1].plot(sorted_abs, cdf)
for threshold in [0.5, 1.0, 2.0, 5.0]:
    frac = (df_match['abs_diff'] <= threshold).mean()
    axes[1].axvline(threshold, color='gray', linestyle=':', alpha=0.5)
    axes[1].text(threshold + 0.1, frac - 0.05, f'{frac:.1%}', fontsize=9)
axes[1].set_xlabel('|OPAM2 pKa - Marvin pKa|')
axes[1].set_ylabel('Cumulative fraction')
axes[1].set_title('CDF of absolute pKa differences')
axes[1].set_xlim(0, 15)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'pka_difference_distribution.png', dpi=150)
plt.show()

# Summary statistics
print(f"Mean signed diff: {df_match['diff'].mean():.2f}")
print(f"Std of diff: {df_match['diff'].std():.2f}")
print(f"Fraction within 0.5 pKa units: {(df_match['abs_diff'] <= 0.5).mean():.1%}")
print(f"Fraction within 1.0 pKa units: {(df_match['abs_diff'] <= 1.0).mean():.1%}")
print(f"Fraction within 2.0 pKa units: {(df_match['abs_diff'] <= 2.0).mean():.1%}")
print(f"Fraction > 5.0 pKa units: {(df_match['abs_diff'] > 5.0).mean():.1%}")

## 3. Categorize pKa changes by magnitude

In [ ]:
def categorize_change(abs_diff):
    if abs_diff < 0.5:
        return 'Small (<0.5)'
    elif abs_diff < 2.0:
        return 'Moderate (0.5-2.0)'
    elif abs_diff < 5.0:
        return 'Large (2.0-5.0)'
    else:
        return 'Very large (>5.0)'

df_match['change_category'] = df_match['abs_diff'].apply(categorize_change)

cat_order = ['Small (<0.5)', 'Moderate (0.5-2.0)', 'Large (2.0-5.0)', 'Very large (>5.0)']
cat_counts = df_match['change_category'].value_counts().reindex(cat_order)
cat_pcts = cat_counts / len(df_match) * 100

print('pKa change magnitude categories:')
for cat in cat_order:
    print(f'  {cat:25s}: {cat_counts[cat]:6,d} ({cat_pcts[cat]:.1f}%)')

## 4. Breakdown by acid vs base type

In [ ]:
for atype, label in [('A', 'Acid'), ('B', 'Base')]:
    sub = df_match[df_match['type'] == atype]
    stats = summary['by_type'].get(atype, {})
    print(f'\n{label} predictions (n={len(sub):,}):')
    print(f'  R² = {stats.get("r2", "n/a")}')
    print(f'  RMSE = {stats.get("rmse", "n/a")}')
    print(f'  MAE = {stats.get("mae", "n/a")}')
    print(f'  Median |diff| = {sub["abs_diff"].median():.2f}')
    print(f'  % within 1.0 = {(sub["abs_diff"] <= 1.0).mean():.1%}')
    print(f'  % > 5.0 = {(sub["abs_diff"] > 5.0).mean():.1%}')

## 5. Breakdown by functional group

In [ ]:
fg_stats = []
for fg, sub in df_match.groupby('functional_group'):
    fg_stats.append({
        'functional_group': fg,
        'n': len(sub),
        'mae': sub['abs_diff'].mean(),
        'median_abs_diff': sub['abs_diff'].median(),
        'pct_within_1': (sub['abs_diff'] <= 1.0).mean() * 100,
        'pct_gt_5': (sub['abs_diff'] > 5.0).mean() * 100,
    })

df_fg = pd.DataFrame(fg_stats).sort_values('n', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_fg['functional_group'], df_fg['mae'])
for bar, n in zip(bars, df_fg['n']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'n={n:,}', va='center', fontsize=9)
ax.set_xlabel('MAE (pKa units)')
ax.set_title('OPAM2 vs Marvin MAE by Functional Group')
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'pka_mae_by_functional_group.png', dpi=150)
plt.show()

print(df_fg.to_string(index=False))

## 6. Per-compound summary: compounds with largest changes

In [ ]:
cpd_summary = df_match.groupby('cpd_id').agg(
    n_atoms=('abs_diff', 'count'),
    mean_abs_diff=('abs_diff', 'mean'),
    max_abs_diff=('abs_diff', 'max'),
    median_abs_diff=('abs_diff', 'median'),
).reset_index().sort_values('max_abs_diff', ascending=False)

print(f'Compounds with matched atoms: {len(cpd_summary):,}')
print(f'\nTop 20 compounds by max |pKa diff|:')
print(cpd_summary.head(20).to_string(index=False))

## 7. Parity plot (OPAM2 vs Marvin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = {'A': 'tab:red', 'B': 'tab:blue'}
labels = {'A': 'Acid', 'B': 'Base'}

for atype, color in colors.items():
    sub = df_match[df_match['type'] == atype]
    axes[0].scatter(sub['marvin_pka'], sub['molgpka_pka'], s=2, alpha=0.15,
                    color=color, label=f"{labels[atype]} (n={len(sub):,})")

lo, hi = -5, 25
axes[0].plot([lo, hi], [lo, hi], 'k--', lw=1)
axes[0].set_xlabel('Marvin pKa')
axes[0].set_ylabel('OPAM2 pKa')
axes[0].set_title(f'Parity plot (R²={summary["overall"]["r2"]:.3f})')
axes[0].legend()
axes[0].set_aspect('equal', 'box')
axes[0].set_xlim(lo, hi)
axes[0].set_ylim(lo, hi)

# Density heatmap
axes[1].hexbin(df_match['marvin_pka'], df_match['molgpka_pka'],
               gridsize=50, mincnt=1, cmap='viridis')
axes[1].plot([lo, hi], [lo, hi], 'w--', lw=1)
axes[1].set_xlabel('Marvin pKa')
axes[1].set_ylabel('OPAM2 pKa')
axes[1].set_title('Density heatmap')
axes[1].set_xlim(lo, hi)
axes[1].set_ylim(lo, hi)
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Count')

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'pka_parity_plot.png', dpi=150)
plt.show()

## 8. Estimate thermodynamic impact magnitude

The Legendre transform in dGPredictor uses pKa values as:
```
dG0s = -cumsum([0] + pKas) * R * T * ln(10)
```
where `R * T * ln(10) = 5.71 kJ/mol` at 298.15 K.

A 1 pKa unit change shifts the cumulative dG0 of that microspecies by ~5.7 kJ/mol (~1.36 kcal/mol).
Changes propagate to all lower-protonation microspecies.

In [ ]:
R = 8.31446e-3  # kJ/(mol*K)
T = 298.15      # K
RT_ln10 = R * T * np.log(10)

df_match['est_dg_shift_kj'] = df_match['abs_diff'] * RT_ln10

print(f'R*T*ln(10) at 298.15 K = {RT_ln10:.3f} kJ/mol')
print(f'\nEstimated per-atom dG shift from pKa changes:')
print(f'  Mean:   {df_match["est_dg_shift_kj"].mean():.2f} kJ/mol')
print(f'  Median: {df_match["est_dg_shift_kj"].median():.2f} kJ/mol')
print(f'  Max:    {df_match["est_dg_shift_kj"].max():.2f} kJ/mol')
print(f'  % > 5 kJ/mol:  {(df_match["est_dg_shift_kj"] > 5).mean():.1%}')
print(f'  % > 10 kJ/mol: {(df_match["est_dg_shift_kj"] > 10).mean():.1%}')

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df_match['est_dg_shift_kj'], bins=80, edgecolor='none', alpha=0.7)
ax.axvline(5, color='orange', linestyle='--', label='5 kJ/mol')
ax.axvline(10, color='red', linestyle='--', label='10 kJ/mol')
ax.set_xlabel('Estimated |dG shift| per atom (kJ/mol)')
ax.set_ylabel('Count')
ax.set_title('Estimated thermodynamic impact of pKa changes')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'estimated_dg_shift.png', dpi=150)
plt.show()

## 9. Coverage analysis: OPAM2 vs current Marvin coverage

In [ ]:
# Count total compounds in ModelSEEDDatabase
import glob

cpd_files = sorted(glob.glob(str(MSDB_ROOT / 'Biochemistry' / 'compound_*.tsv')))
cpd_files = [f for f in cpd_files if 'provenance' not in f]

all_cpds = pd.concat([pd.read_csv(f, sep='\t', usecols=['id', 'pka', 'pkb']) for f in cpd_files])
has_pka = all_cpds['pka'].notna() & (all_cpds['pka'] != '')
has_pkb = all_cpds['pkb'].notna() & (all_cpds['pkb'] != '')
has_any = has_pka | has_pkb

mol_files = set(p.stem for p in (OPAM2_ROOT / 'src' / 'datasets' / 'Unique_MS_Structures').glob('cpd*.mol'))

print(f'Total compounds in ModelSEEDDatabase: {len(all_cpds):,}')
print(f'Compounds with Marvin pKa: {has_pka.sum():,}')
print(f'Compounds with Marvin pkb: {has_pkb.sum():,}')
print(f'Compounds with any Marvin data: {has_any.sum():,}')
print(f'Compounds with .mol files (OPAM2 can predict): {len(mol_files):,}')
print(f'Compounds with .mol AND Marvin data: {sum(1 for cid in all_cpds[has_any]["id"] if cid in mol_files):,}')

## 10. Save processed comparison data for downstream notebooks

In [ ]:
df_match.to_csv(DATA_DIR / 'pka_comparison.tsv', sep='\t', index=False)
cpd_summary.to_csv(DATA_DIR / 'pka_per_compound_summary.tsv', sep='\t', index=False)
df_fg.to_csv(DATA_DIR / 'pka_functional_group_stats.tsv', sep='\t', index=False)

print(f'Saved {len(df_match):,} atom-level comparisons to data/pka_comparison.tsv')
print(f'Saved {len(cpd_summary):,} compound summaries to data/pka_per_compound_summary.tsv')
print(f'Saved {len(df_fg)} functional group stats to data/pka_functional_group_stats.tsv')